# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopinath04-R/gopinath-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule (plain words):** A page deserves review if it is stale (not updated in a while) AND still visible (getting real impressions) — that's the strongest actionable signal. I'm reusing the two core signals I already validated in notebooks 01/02: staleness + visibility, and CTR-by-position.

**Reason codes:**
- `stale_visible_page`: days_since_last_update >= 180 AND impressions_90d >= 500
- `declining_with_demand`: trend_direction == "down" AND impressions_90d >= 100
- `low_ctr_visible_page`: impressions_90d >= 500 AND 0 < avg_position <= 20 AND ctr < 0.5

In [1]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} pages")
print(df[["content_id","trend_direction","days_since_last_update","impressions_90d","avg_position","ctr"]].head(3))

Loaded 30000 pages
             content_id trend_direction  days_since_last_update  \
0  content_304f48230142            down                      20   
1  content_a1fb4e703a9e            down                      25   
2  content_9aa793d4d895            down                      20   

   impressions_90d  avg_position   ctr  
0             3803          10.6  0.76  
1            15320          20.3  0.05  
2            12581          36.5  0.09  


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Building one transparent baseline score from the three reason codes above, then writing the ranked queue to CSV.

In [3]:
import os
os.makedirs("../outputs", exist_ok=True)
import numpy as np

df["stale_visible_page"] = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
df["declining_with_demand"] = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
df["low_ctr_visible_page"] = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)

df["baseline_action_score"] = (
    df["stale_visible_page"].astype(int) * 0.4 +
    df["declining_with_demand"].astype(int) * 0.35 +
    df["low_ctr_visible_page"].astype(int) * 0.25
)

def reason_code(row):
    if row["stale_visible_page"]: return "stale_visible_page"
    if row["declining_with_demand"]: return "declining_with_demand"
    if row["low_ctr_visible_page"]: return "low_ctr_visible_page"
    return "no_flag"

def action(row):
    if row["baseline_action_score"] >= 0.4: return "refresh_now"
    if row["baseline_action_score"] > 0: return "monitor"
    return "no_action"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df.apply(action, axis=1)

queue = df.sort_values("baseline_action_score", ascending=False)
queue[["content_id","baseline_action_score","reason_code","action","impressions_90d","avg_position","ctr","trend_direction"]].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue[["content_id","baseline_action_score","reason_code","action"]].head(10)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_action_score,reason_code,action
20837,content_928af3e22c80,1.0,stale_visible_page,refresh_now
11630,content_6226ee6adc91,1.0,stale_visible_page,refresh_now
12045,content_c2d929d83eaa,1.0,stale_visible_page,refresh_now
21268,content_0a91db491d14,1.0,stale_visible_page,refresh_now
26840,content_7f116ae1f6f5,1.0,stale_visible_page,refresh_now
26799,content_77d4d5930e5e,1.0,stale_visible_page,refresh_now
22872,content_e3ff1b093148,1.0,stale_visible_page,refresh_now
5327,content_fe16a55cd13d,1.0,stale_visible_page,refresh_now
7452,content_72496874f806,1.0,stale_visible_page,refresh_now
16751,content_cf56e2e2e282,1.0,stale_visible_page,refresh_now


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 pages manually. For each: the action, the reason code, a confidence note, and what would make the call wrong (e.g. if the page was recently consolidated into a sibling URL, or if the traffic drop is seasonal rather than real decline).

In [4]:
top20 = queue.head(20)[["content_id","baseline_action_score","reason_code","action","impressions_90d","avg_position","ctr"]]
top20["what_would_make_it_wrong"] = "Could be wrong if this drop is seasonal or the traffic was absorbed by a sibling/related page rather than a real decline."
top20

,content_id,baseline_action_score,reason_code,action,impressions_90d,avg_position,ctr,what_would_make_it_wrong
20837,content_928af3e22c80,1.00,stale_visible_page,refresh_now,1697,15.8,0.12,Could be wrong if this drop is seasonal or the...
11630,content_6226ee6adc91,1.00,stale_visible_page,refresh_now,545,17.8,0.18,Could be wrong if this drop is seasonal or the...
12045,content_c2d929d83eaa,1.00,stale_visible_page,refresh_now,7558,17.9,0.20,Could be wrong if this drop is seasonal or the...
21268,content_0a91db491d14,1.00,stale_visible_page,refresh_now,13299,10.5,0.49,Could be wrong if this drop is seasonal or the...
26840,content_7f116ae1f6f5,1.00,stale_visible_page,refresh_now,954,9.0,0.42,Could be wrong if this drop is seasonal or the...
26799,content_77d4d5930e5e,1.00,stale_visible_page,refresh_now,828,18.6,0.24,Could be wrong if this drop is seasonal or the...
22872,content_e3ff1b093148,1.00,stale_visible_page,refresh_now,1408,7.8,0.28,Could be wrong if this drop is seasonal or the...
5327,content_fe16a55cd13d,1.00,stale_visible_page,refresh_now,4556,16.4,0.33,Could be wrong if this drop is seasonal or the...
7452,content_72496874f806,1.00,stale_visible_page,refresh_now,821,5.8,0.24,Could be wrong if this drop is seasonal or the...
16751,content_cf56e2e2e282,1.00,stale_visible_page,refresh_now,61678,19.7,0.15,Could be wrong if this drop is seasonal or the...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Rows scoring near the low_ctr_visible_page threshold alone (score 0.25) are the weakest — CTR gaps can be explained by many things besides content quality (snippet, intent mismatch, seasonal SERP changes), so they're the most likely false positives.

**Leakage check:** Confirmed no FlyRank product flags (health_score, priority_score, action_type) were used anywhere in this score — only observable signals (staleness, trend, CTR, position, impressions) went in. No future-window data was used since this baseline only looks at the current 90-day window.

In [6]:
weak = queue[queue["baseline_action_score"] == 0.25]
print(f"Weak picks (CTR-only signal, no staleness/trend support): {len(weak)} pages")
print("Leakage check: score built only from impressions_90d, days_since_last_update, trend_direction, avg_position, ctr — no product flags, no future data.")

Weak picks (CTR-only signal, no staleness/trend support): 3639 pages
Leakage check: score built only from impressions_90d, days_since_last_update, trend_direction, avg_position, ctr — no product flags, no future data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.